# Hadronic-channel analysis

This notebook performs generator-level studies of the hadronic \(W^+W^-\) channel. It reconstructs W candidates from four final-state quarks/gluons, builds luminosity-normalized kinematic templates, performs binned likelihood fits and scans, and trains a binary signal/background BDT.

## Input data and requirements

The notebook requires the four hadronic signal samples and one hadronic background sample configured in `HADRONIC_SAMPLE_FILES` in `analysis_config.py`. The standard input tree is `Events` (configurable through `FCC_EVENTS_TREE`) and must contain the LHE particle branches listed in the repository README.

ROOT with PyROOT is required. The BDT sections also require ROOT with TMVA support and the Python packages in `requirements.txt`.

## Event selection and observables

Events are selected when exactly four final-state quarks/gluons with `pT > 0.1 GeV` are available. The three possible pairings into two W candidates are tested, and the pairing closest to the configured W-boson mass is selected. The principal observables are the absolute decay-angle cosine, `|cos(theta*)|`, and the transverse momentum of the leading parton.

For the BDT workflow, the event split is deterministic: 70% of each source file (entry indices with `i % 10 < 7`) are used for training, and the remaining 30% are reserved for holdout templates. The kinematic distributions use all selected events.

## How to run and outputs

Update `HADRONIC_SAMPLE_FILES` and the hadronic cross-sections in `analysis_config.py`, then execute all cells from top to bottom. The notebook writes `outputs/histos_hadronic.root`, intermediate TMVA files and `outputs/TMVA_Adronico_Output.root`. TMVA weights are stored in `dataset_adr/weights/`; plots are saved under `outputs/plots/`.


In [ ]:
# 1. Setup and configuration
import array
import math
import os
from itertools import combinations

import numpy as np
from iminuit import Minuit

# PyROOT must be available in the active Python environment.
import ROOT

from analysis_config import (
    BINS_BDT,
    COLLISION_ENERGY_GEV,
    HADRONIC_CROSS_SECTIONS_PB,
    HADRONIC_SAMPLE_FILES,
    HADRONIC_TEMPLATE_FILE,
    EVENTS_TREE_NAME,
    LUMI_PB,
    N_BINS,
    PROJECT_DIR,
    REBIN_BDT,
    W_MASS_GEV as M_W,
    output_path,
    resolve_input,
)

# Keep TMVA's relative dataset/weights directory inside the project.
os.chdir(PROJECT_DIR)

ROOT.TMVA.Tools.Instance()

# Keep ROOT graphics objects alive for the following cells.
ROOT_KEEPALIVE = []

# Input sample names are configured centrally in analysis_config.py.
FILE_00 = HADRONIC_SAMPLE_FILES["00"]
FILE_TT = HADRONIC_SAMPLE_FILES["TT"]
FILE_0T = HADRONIC_SAMPLE_FILES["0T"]
FILE_T0 = HADRONIC_SAMPLE_FILES["T0"]
FILE_BKG = HADRONIC_SAMPLE_FILES["background"]

# Physics settings are shared through analysis_config.py.
SIGMA_00 = HADRONIC_CROSS_SECTIONS_PB["00"]
SIGMA_TT = HADRONIC_CROSS_SECTIONS_PB["TT"]
SIGMA_0T = HADRONIC_CROSS_SECTIONS_PB["0T"]
SIGMA_T0 = HADRONIC_CROSS_SECTIONS_PB["T0"]
SIGMA_BKG = HADRONIC_CROSS_SECTIONS_PB["background"]

In [ ]:
# Shared plot styling and ROOT graphics helpers.
# Text and line sizes are fractions of the pad height. Adjust them here
# to change the readability of the exported figures.
SIZE_LABEL  = 0.045     # numeri sugli assi
SIZE_TITLE  = 0.050     # titoli degli assi
SIZE_LEGEND = 0.036     # testo delle legende
SIZE_FCC    = 0.045     # scritta "FCC-ee Simulation" (l'energia e' un po' piu' piccola)
LINE_WIDTH  = 3         # spessore delle linee degli istogrammi

# --- ROOT global settings ---
ROOT.TH1.AddDirectory(False)        # gli istogrammi non dipendono dai file aperti
ROOT.gStyle.SetOptStat(0)           # niente riquadro delle statistiche
ROOT.gStyle.SetOptTitle(0)          # niente titolo in cima ai grafici
ROOT.gStyle.SetPadTickX(1)          # tacche anche sull'asse superiore
ROOT.gStyle.SetPadTickY(1)          # tacche anche sull'asse destro
ROOT.gStyle.SetEndErrorSize(0)      # barre d'errore senza cappuccio
ROOT.gStyle.SetFrameLineWidth(2)    # assi piu' spessi
ROOT.gStyle.SetLabelFont(42, "xyz")
ROOT.gStyle.SetTitleFont(42, "xyz")
ROOT.gStyle.SetLabelSize(SIZE_LABEL, "xyz")
ROOT.gStyle.SetTitleSize(SIZE_TITLE, "xyz")
ROOT.gStyle.SetTitleOffset(1.4, "xy")

# --- Sample colours ---
COLORS = {
    "00":    ROOT.TColor.GetColor("#0072B2"),   # blu     -> longitudinale (LL)
    "TT":    ROOT.TColor.GetColor("#C0392B"),   # rosso   -> trasversa (TT)
    "mixed": ROOT.TColor.GetColor("#7B3294"),   # viola   -> mista (LT)
    "bkg":   ROOT.TColor.GetColor("#666666"),   # grigio  -> fondo
    "sig":   ROOT.TColor.GetColor("#000000"),   # nero    -> segnale totale
}

# --- Colours for likelihood fits and scans ---
COLORS_FIT = {
    "profilato":        ROOT.TColor.GetColor("#2171B5"),   # altri parametri profilati
    "profilato_chiaro": ROOT.TColor.GetColor("#9ECAE1"),
    "fisso":            ROOT.TColor.GetColor("#000000"),   # altri parametri fissi a 1
    "fisso_chiaro":     ROOT.TColor.GetColor("#808080"),
    "soglie":           ROOT.TColor.GetColor("#808080"),   # linee a 1 e 2 sigma
}

# --- Etichette ---
Y_TITLE = "Events / bin"
LABELS = {
    "00": "Longitudinal (00)",
    "TT": "Transverse (TT)",
    "mixed": "Mixed (0T + T0)",
    "bkg": "Background",
    "sig": "Signal",
}


# --- Funzioni grafiche ---
def make_canvas(name, title="", width=900, height=650):
    """Create a canvas with the margins used for the analysis plots."""
    c = ROOT.TCanvas(name, title, width, height)
    c.SetLeftMargin(0.17)
    c.SetBottomMargin(0.18)
    c.SetTopMargin(0.12)
    c.SetRightMargin(0.05)
    return c


# Keep TLatex objects alive while the canvases are displayed.
_FCC_LABEL_OBJECTS = []

def draw_fcc_label():
    """Draw the FCC-ee Simulation label, collision energy, and integrated luminosity."""
    latex = ROOT.TLatex()
    latex.SetNDC()
    latex.SetTextFont(42)
    latex.SetTextAlign(11)                  # allineato a sinistra
    latex.SetTextSize(SIZE_FCC)
    # Use one string with two fonts: #font[61] is bold and #font[52] is italic.
    t1 = latex.DrawLatex(0.24, 0.92, "#font[61]{FCC-ee} #font[52]{#scale[0.85]{Simulation}}")
    latex.SetTextAlign(31)                  # allineato a destra, sul bordo del grafico
    latex.SetTextSize(SIZE_FCC * 0.78)
    t2 = latex.DrawLatex(0.95, 0.92, f"#sqrt{{s}} = {COLLISION_ENERGY_GEV:g} GeV, L = {LUMI_PB / 1e6:g} ab^{{-1}}")
    _FCC_LABEL_OBJECTS.extend([latex, t1, t2])


def style_hist(hist, color, line_style=1, fill=True):
    """Set the histogram line colour and width, with an optional transparent fill."""
    hist.SetLineColor(color)
    hist.SetLineWidth(LINE_WIDTH)
    hist.SetLineStyle(line_style)
    if fill:
        hist.SetFillColorAlpha(color, 0.15)


def make_legend(x1, y1, x2, y2, header=None):
    """Create a legend without a border or background, with an optional header."""
    leg = ROOT.TLegend(x1, y1, x2, y2)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextFont(42)
    leg.SetTextSize(SIZE_LEGEND)
    leg.SetMargin(0.12)                     # meno spazio per il simbolo, piu' per il testo
    if header:
        leg.SetHeader(header)
    return leg


## 1. Luminosity-normalized kinematic distributions 

In [ ]:
# 2. Read events and calculate the observables from the four partons

def open_tree(filename):
    f = ROOT.TFile.Open(resolve_input(filename), "READ")
    if not f or f.IsZombie():
        raise FileNotFoundError(f"Impossibile aprire il file: {filename}")
    return f, f.Get(EVENTS_TREE_NAME)


def get_partons(event):
    """Return the final-state quarks and gluons in an LHE event as TLorentzVectors."""
    partons = []
    for p in range(event.nLHEPart):
        if abs(event.LHEPart_pdgId[p]) in (1, 2, 3, 4, 5, 21) and event.LHEPart_pt[p] > 0.1:
            v = ROOT.TLorentzVector()
            v.SetPtEtaPhiM(event.LHEPart_pt[p], event.LHEPart_eta[p],
                           event.LHEPart_phi[p], event.LHEPart_mass[p])
            partons.append(v)
    return partons


def cos_theta_star(parent, daughter):
    """Calculate the daughter angle in the parent rest frame, relative to the parent flight direction."""
    d_rest = ROOT.TLorentzVector(daughter)
    d_rest.Boost(-parent.BoostVector())

    p_d = d_rest.Vect()
    p_w = parent.Vect()

    denom = p_d.Mag() * p_w.Mag()
    if denom < 1e-10:
        return None

    cos_theta = p_d.Dot(p_w) / denom
    return max(-1.0, min(1.0, cos_theta))


def calcola_osservabili(partons):
    """Calculate |cos(theta*)| and the transverse momentum of the leading parton from four partons.

       1. Si provano i 3 possibili abbinamenti in due coppie e si sceglie
          quello con le masse piu' vicine a M_W.
       2. Il W scelto e' quello che contiene il partone con pT massimo, e il
          "figlio" e' proprio quel partone (nessuna scelta casuale: stesso evento
          -> stesso risultato). Con |cos(theta*)| scegliere uno o l'altro dei due
          figli non cambia nulla, perche' nel sistema di riposo del W sono opposti.
    """

    migliore = None

    for a, b, c, d in ((0, 1, 2, 3), (0, 2, 1, 3), (0, 3, 1, 2)):
        w1 = partons[a] + partons[b]
        w2 = partons[c] + partons[d]
        distanza = math.sqrt((w1.M() - M_W)**2 + (w2.M() - M_W)**2)
        if migliore is None or distanza < migliore[0]:
            migliore = (distanza, w1, w2, a, b)

    distanza, w1, w2, a, b = migliore

    if w1.M() < 1 or w2.M() < 1 or w1.P() < 0.1 or w2.P() < 0.1:
        return None

    # Index of the parton with the highest pT.
    i_lead = max(range(4), key=lambda i: partons[i].Pt())
    jet_pt = partons[i_lead].Pt()

    # w1 is formed from partons a and b; choose w1 if it contains the leading parton.
    if i_lead in (a, b):
        w_scelto = w1
    else:
        w_scelto = w2

    cos_star = cos_theta_star(w_scelto, partons[i_lead])
    if cos_star is None:
        return None

    return min(abs(cos_star), 1.0), jet_pt


def leggi_campione(filename, sigma):
    """Read a sample file and return a dictionary containing the calculated event observables.

    Il 70% degli eventi (indice i%10 < 7) e' per il training del BDT, il 30% resta da parte
    ("holdout") to build the templates. The first set of histograms uses all selected events."""
    f, tree = open_tree(filename)
    n_gen = int(tree.GetEntries())
    train = []
    holdout = []
    n_holdout = 0                        # eventi holdout contati PRIMA di qualsiasi selezione

    for i, event in enumerate(tree):
        is_holdout = (i % 10 >= 7)
        if is_holdout:
            n_holdout += 1
        partons = get_partons(event)
        if len(partons) != 4:
            continue
        osservabili = calcola_osservabili(partons)
        if osservabili is None:
            continue
        if is_holdout:
            holdout.append(osservabili)
        else:
            train.append(osservabili)

    f.Close()
    print(f"{filename}: {n_gen} generated events, {len(train) + len(holdout)} selected events")
    return {"sigma": sigma, "n_gen": n_gen, "n_holdout": n_holdout, "train": train, "holdout": holdout}


campioni = {
    "00": leggi_campione(FILE_00, SIGMA_00),
    "TT": leggi_campione(FILE_TT, SIGMA_TT),
    "0T": leggi_campione(FILE_0T, SIGMA_0T),
    "T0": leggi_campione(FILE_T0, SIGMA_T0),
    "bkg": leggi_campione(FILE_BKG, SIGMA_BKG),
}

In [ ]:
# 3. Histograms normalized to the integrated luminosity:
# Signal components versus background.
vars_list = ["cos_theta_star", "jet_pt"]
ranges = {
    "cos_theta_star": (0.0, 1.0),
    "jet_pt": (0.0, 90.0)
}
titoli = {
    "cos_theta_star": "|cos(#theta^{*})|",
    "jet_pt": "Leading parton p_{T} [GeV]"
}
# --- Filling and normalization ---
h_dict = {}
for k, var in enumerate(vars_list):
    x_min, x_max = ranges[var]
    for nome in ["00", "TT", "0T", "T0", "bkg"]:
        h = ROOT.TH1F(
            f"h_{var}_{nome}",
            f";{titoli[var]};{Y_TITLE}",
            N_BINS,
            x_min,
            x_max
        )
        h.Sumw2()
        c = campioni[nome]
        for evento in c["train"] + c["holdout"]:
            h.Fill(evento[k])
        h.Scale(c["sigma"] * LUMI_PB / c["n_gen"])
        h_dict[f"{nome}_{var}"] = h
    # Combine the mixed polarization components.
    h_mixed = h_dict[f"0T_{var}"].Clone(f"h_{var}_mixed")
    h_mixed.Add(h_dict[f"T0_{var}"])
    h_dict[f"mixed_{var}"] = h_mixed

# --- Plots ---
LABELS["bkg"] = "Background (e^{+}e^{-} #rightarrow jjjj)"
canvases = []
legends = []

for var in vars_list:
    c = make_canvas(f"c_comp_{var}", f"Components vs Bkg - {var}")
    c.cd()

    h_00 = h_dict[f"00_{var}"]
    h_tt = h_dict[f"TT_{var}"]
    h_mix = h_dict[f"mixed_{var}"]
    h_bkg = h_dict[f"bkg_{var}"]
    istogrammi = [(h_00, "00"), (h_tt, "TT"), (h_mix, "mixed"), (h_bkg, "bkg")]

    for h, chiave in istogrammi:
        style_hist(h, COLORS[chiave], fill=(chiave != "bkg"))

    max_val = max(h.GetMaximum() for h, chiave in istogrammi)
    h_00.GetYaxis().SetRangeUser(0.0, max_val * 1.50)
    h_00.GetYaxis().SetTitleOffset(1.4)
    h_00.Draw("HIST E")
    h_tt.Draw("HIST E SAME")
    h_mix.Draw("HIST E SAME")
    h_bkg.Draw("HIST E SAME")

    leg = make_legend(0.20, 0.66, 0.95, 0.85)
    for h, chiave in istogrammi:
        leg.AddEntry(h, f"{LABELS[chiave]} [Int: {h.Integral():.3e}]", "l")
    leg.Draw()

    draw_fcc_label()

    c.Modified()
    c.Update()
    c.Draw()

    # Keep each canvas and legend inside the loop for its corresponding observable.
    canvases.append(c)
    legends.append(leg)

# --- Save histograms ---
f_out = ROOT.TFile.Open(HADRONIC_TEMPLATE_FILE, "RECREATE")
f_out.cd()
for var in vars_list:
    h_dict[f"00_{var}"].Write(f"h_{var}_00")
    h_dict[f"TT_{var}"].Write(f"h_{var}_tt")
    h_dict[f"mixed_{var}"].Write(f"h_{var}_mixed")
    h_dict[f"bkg_{var}"].Write(f"h_{var}_bkg")
f_out.Close()
print("Salvataggio in 'histos_adronico.root'.")

In [ ]:
# ============================================================
# 4. Study the decay-angle observable |cos(theta*)| (charge combined)
# Overlay and stacked plots for the hadronic channel.
# ============================================================

TITLE_ABS_COS_THETA_STAR = "|cos(#theta^{*})|"

# --- Retrieve the histograms from h_dict ---
h_abs_00 = h_dict["00_cos_theta_star"].Clone("h_abs_00")
h_abs_tt = h_dict["TT_cos_theta_star"].Clone("h_abs_tt")
h_abs_mixed = h_dict["mixed_cos_theta_star"].Clone("h_abs_mixed")

# Set the axis titles.
for h in [h_abs_00, h_abs_tt, h_abs_mixed]:
    h.GetXaxis().SetTitle(TITLE_ABS_COS_THETA_STAR)
    h.GetYaxis().SetTitle(Y_TITLE)
    h.SetStats(False)

# ============================================================
# A. Signal-component overlay.
# ============================================================
c_sig_comp = make_canvas("c_sig_comp_abs", "Signal Components |cos(theta*)| - Lines")
c_sig_comp.cd()

for h, col in zip([h_abs_00, h_abs_tt, h_abs_mixed], [COLORS["00"], COLORS["TT"], COLORS["mixed"]]):
    style_hist(h, col)

max_val_sig = max(h_abs_00.GetMaximum(), h_abs_tt.GetMaximum(), h_abs_mixed.GetMaximum())
h_abs_00.GetYaxis().SetRangeUser(0.0, max_val_sig * 1.45)
h_abs_00.GetYaxis().SetTitleOffset(1.4)

h_abs_00.Draw("HIST E")
h_abs_tt.Draw("HIST E SAME")
h_abs_mixed.Draw("HIST E SAME")

leg_sig_comp = make_legend(0.20, 0.72, 0.95, 0.88)
leg_sig_comp.AddEntry(h_abs_00, f"{LABELS['00']}  [Integral = {h_abs_00.Integral():.4g}]", "l")
leg_sig_comp.AddEntry(h_abs_tt, f"{LABELS['TT']}  [Integral = {h_abs_tt.Integral():.4g}]", "l")
leg_sig_comp.AddEntry(h_abs_mixed, f"{LABELS['mixed']}  [Integral = {h_abs_mixed.Integral():.4g}]", "l")
leg_sig_comp.Draw()

draw_fcc_label()
c_sig_comp.Draw()
canvases.append(c_sig_comp)
legends.append(leg_sig_comp)


# ============================================================
# B. Stacked signal distribution (|cos(theta*)|).
# ============================================================
c_stack_sig = make_canvas("c_stack_sig_abs", "Signal Components |cos(theta*)| - Stacked")
c_stack_sig.cd()

stack_sig = ROOT.THStack("stack_sig_abs", f";{TITLE_ABS_COS_THETA_STAR};{Y_TITLE}")
stack_sig.Add(h_abs_00)
stack_sig.Add(h_abs_mixed)
stack_sig.Add(h_abs_tt)

h_total_sig = h_abs_00.Clone("h_total_sig_abs")
h_total_sig.Add(h_abs_mixed)
h_total_sig.Add(h_abs_tt)
h_total_sig.SetFillStyle(3004)
h_total_sig.SetFillColor(ROOT.kGray + 2)
h_total_sig.SetLineColor(ROOT.kGray + 2)
h_total_sig.SetMarkerSize(0)
h_total_sig.SetStats(False)

stack_sig.Draw("HIST")
stack_sig.GetXaxis().SetTitle(TITLE_ABS_COS_THETA_STAR)
stack_sig.GetYaxis().SetTitle(Y_TITLE)
stack_sig.GetYaxis().SetTitleOffset(1.4)
stack_sig.SetMinimum(0)
stack_sig.SetMaximum(stack_sig.GetMaximum() * 1.45)

h_total_sig.Draw("E2 SAME")

leg_stack_sig = make_legend(0.20, 0.68, 0.95, 0.88)
leg_stack_sig.AddEntry(h_abs_tt, f"{LABELS['TT']}  [Integral = {h_abs_tt.Integral():.4g}]", "lf")
leg_stack_sig.AddEntry(h_abs_mixed, f"{LABELS['mixed']}  [Integral = {h_abs_mixed.Integral():.4g}]", "lf")
leg_stack_sig.AddEntry(h_abs_00, f"{LABELS['00']}  [Integral = {h_abs_00.Integral():.4g}]", "lf")
leg_stack_sig.AddEntry(h_total_sig, f"MC Stat. Uncertainty  [Total integral = {h_total_sig.Integral():.4g}]", "f")
leg_stack_sig.Draw()

draw_fcc_label()
c_stack_sig.Draw()
canvases.append(c_stack_sig)
legends.append(leg_stack_sig)

In [ ]:

# ============================================================
# 5. Study the leading-parton transverse momentum.
# Overlay of the signal components.
# ============================================================

TITLE_LEADING_PT = "Leading parton p_{T} [GeV]"

# --- Retrieve histograms from h_dict ---
h_pt_00 = h_dict["00_jet_pt"].Clone("h_pt_00")
h_pt_tt = h_dict["TT_jet_pt"].Clone("h_pt_tt")
h_pt_mixed = h_dict["mixed_jet_pt"].Clone("h_pt_mixed")

# Set the axis titles.
for h in [h_pt_00, h_pt_tt, h_pt_mixed]:
    h.GetXaxis().SetTitle(TITLE_LEADING_PT)
    h.GetYaxis().SetTitle(Y_TITLE)
    h.SetStats(False)


# ============================================================
# Signal-component overlay.
# ============================================================

c_pt_comp = make_canvas(
    "c_pt_comp",
    "Signal Components Leading parton pT - Lines"
)

c_pt_comp.cd()

# Apply the plot style defined above.
for h, col in zip(
    [h_pt_00, h_pt_tt, h_pt_mixed],
    [COLORS["00"], COLORS["TT"], COLORS["mixed"]]
):
    style_hist(h, col)

# Set the y-axis range.
max_val_pt = max(
    h_pt_00.GetMaximum(),
    h_pt_tt.GetMaximum(),
    h_pt_mixed.GetMaximum()
)

h_pt_00.GetYaxis().SetRangeUser(0.0, max_val_pt * 1.45)
h_pt_00.GetYaxis().SetTitleOffset(1.4)

# Draw the three signal components.
h_pt_00.Draw("HIST E")
h_pt_tt.Draw("HIST E SAME")
h_pt_mixed.Draw("HIST E SAME")

# Legend.
leg_pt_comp = make_legend(0.20, 0.72, 0.95, 0.88)

leg_pt_comp.AddEntry(
    h_pt_00,
    f"{LABELS['00']} [Integral = {h_pt_00.Integral():.4g}]",
    "l"
)

leg_pt_comp.AddEntry(
    h_pt_tt,
    f"{LABELS['TT']} [Integral = {h_pt_tt.Integral():.4g}]",
    "l"
)

leg_pt_comp.AddEntry(
    h_pt_mixed,
    f"{LABELS['mixed']} [Integral = {h_pt_mixed.Integral():.4g}]",
    "l"
)

leg_pt_comp.Draw()

# FCC-ee label.
draw_fcc_label()

# Update the canvas.
c_pt_comp.Modified()
c_pt_comp.Update()
c_pt_comp.Draw()

# Keep the ROOT objects alive.
canvases.append(c_pt_comp)
legends.append(leg_pt_comp)


## 2. Binned likelihood fits and likelihood scans

The fit uses the hadronic templates produced in the first section. It estimates the three signal-strength parameters for the longitudinal, transverse, and mixed components, using Asimov data formed from the nominal templates.

In [ ]:
# 4. One-, two-, and three-parameter likelihood fits
PARAMS = ("mu_00", "mu_tt", "mu_mix")
VARIABILI = ("cos_theta_star", "jet_pt")

NOMI_PARAMETRI = {"mu_00": "μ_LL", "mu_tt": "μ_TT", "mu_mix": "μ_LT"}
NOMI_VARIABILI = {
    "cos_theta_star": "|cos(θ*)|",
    "jet_pt": "pT del partone con pT massimo",
}


def carica_template(var_name, root_file=HADRONIC_TEMPLATE_FILE):
    """Load ROOT templates and return the background, LL, TT, and mixed components as NumPy arrays."""
    f_input = ROOT.TFile.Open(root_file, "READ")
    if not f_input or f_input.IsZombie():
        raise OSError(f"Non riesco ad aprire il file ROOT: {root_file}")
    template = {}
    for chiave in ("bkg", "00", "tt", "mixed"):
        h = f_input.Get(f"h_{var_name}_{chiave}")
        template[chiave] = np.array([h.GetBinContent(i) for i in range(1, h.GetNbinsX() + 1)], dtype=float)
    f_input.Close()
    return template


def crea_nll(template):
    """Calculate the Poisson deviance (-2 Delta log L) using the sum of the templates as Asimov data."""
    bkg = template["bkg"]
    sig00 = template["00"]
    sigtt = template["tt"]
    sigmix = template["mixed"]

    data = bkg + sig00 + sigtt + sigmix
    mask_data = data > 0

    def negative_log_likelihood(mu_00, mu_tt, mu_mix):
        nu = bkg + mu_00 * sig00 + mu_tt * sigtt + mu_mix * sigmix
        if np.any((nu <= 0) & mask_data):
            return 1e100
        term_data = np.zeros_like(data)
        term_data[mask_data] = data[mask_data] * np.log(data[mask_data] / nu[mask_data])
        return float(2.0 * np.sum(nu - data + term_data))

    return negative_log_likelihood


def esegui_fit_likelihood(template, parametri_liberi=PARAMS):
    """Run the fit, fixing all parameters not selected as free to 1."""
    m = Minuit(crea_nll(template), mu_00=1.0, mu_tt=1.0, mu_mix=1.0)
    m.errordef = 1.0
    for p in PARAMS:
        m.fixed[p] = p not in parametri_liberi
    m.migrad()
    m.hesse()
    return m


def leggi_correlazione(fit, p1, p2, parametri_liberi):
    """Extract a parameter correlation across supported iminuit versions."""
    corr = fit.covariance.correlation()
    try:
        return float(corr[p1, p2])
    except Exception:
        arr = np.asarray(corr, dtype=float)
        ordine = list(PARAMS) if arr.shape[0] == len(PARAMS) else list(parametri_liberi)
        return float(arr[ordine.index(p1), ordine.index(p2)])


fit_results = {}
for var_name in VARIABILI:
    template = carica_template(var_name)
    print(f"\n===== {NOMI_VARIABILI[var_name]} =====")

    for n_liberi in (1, 2, 3):
        for liberi in combinations(PARAMS, n_liberi):
            fit = esegui_fit_likelihood(template, liberi)
            fit_results[(var_name, liberi)] = fit

            riga = f"{n_liberi}D  "
            for p in PARAMS:
                if p in liberi:
                    riga += f"{NOMI_PARAMETRI[p]} = {fit.values[p]:.5f} ± {fit.errors[p]:.5f}   "
                else:
                    riga += f"{NOMI_PARAMETRI[p]} = 1 (fisso)   "
            print(riga + f"| fit valid: {fit.valid}")

    print("3D-fit correlations:")
    for p1, p2 in combinations(PARAMS, 2):
        rho = leggi_correlazione(fit_results[(var_name, PARAMS)], p1, p2, PARAMS)
        print(f"   ρ({NOMI_PARAMETRI[p1]}, {NOMI_PARAMETRI[p2]}) = {rho:+.4f}")

print("\nNote: the Asimov data are the sum of the templates, so each nominal signal strength is 1 by construction.")
print("The relevant result is the expected uncertainty on each parameter.")

In [ ]:
# 5. One- and two-dimensional likelihood scans


VARIABILI_SCAN = VARIABILI         

# Parameter labels in ROOT format (for axes and legends).
LATEX_PARAMETRI = {"mu_00": "#mu_{LL}", "mu_tt": "#mu_{TT}", "mu_mix": "#mu_{LT}"}

tieni_vivi = []     # canvas e grafici vanno tenuti in una lista, altrimenti Python li cancella


def punti_buoni(x, y, ok=None):
    """Remove scan points where the fit failed (large objective value or ok = False)."""
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    buoni = y < 100
    if ok is not None:
        buoni = buoni & np.array(ok, dtype=bool)
    return x[buoni], y[buoni]


# ============================================================
# One-dimensional scan for a single parameter.
# ============================================================
def scan_1d(var_name, param):
    fit_fisso = fit_results[(var_name, (param,))]     # altri parametri fissi a 1
    fit_prof = fit_results[(var_name, PARAMS)]        # altri parametri profilati

    # Use the same range for both curves: plus/minus three standard deviations from the 3D fit.
    val, err = fit_prof.values[param], fit_prof.errors[param]
    intervallo = (val - 3 * err, val + 3 * err)

    # profile: evaluate the function with the other parameters fixed to their nominal value (= 1).
    r = fit_fisso.profile(param, size=40, bound=intervallo, subtract_min=True)
    x_f, y_f = punti_buoni(r[0], r[1])

    # mnprofile: refit at each point while profiling the other two parameters.
    x_p, y_p, ok_p = fit_prof.mnprofile(param, size=40, bound=intervallo, subtract_min=True)
    x_p, y_p = punti_buoni(x_p, y_p, ok_p)

    c = make_canvas(f"c_scan1d_{var_name}_{param}", "Likelihood Scan 1D")
    c.SetTopMargin(0.20)            # spazio sopra il grafico per la legenda
    c.cd()

    g_p = ROOT.TGraph(len(x_p), x_p, y_p)
    g_p.SetTitle(f";{LATEX_PARAMETRI[param]};-2 #Delta ln L")
    g_p.SetLineColor(COLORS_FIT["profilato"])
    g_p.SetLineWidth(LINE_WIDTH)
    g_p.SetMinimum(0)               # asse y da 0 a 9: si vedono le soglie di 1 e 2 sigma
    g_p.SetMaximum(9)
    g_p.Draw("AL")
    g_p.GetXaxis().SetLimits(intervallo[0], intervallo[1])
    g_p.GetXaxis().SetTitleOffset(1.4)
    g_p.GetYaxis().SetTitleOffset(1.4)

    g_f = ROOT.TGraph(len(x_f), x_f, y_f)
    g_f.SetLineColor(COLORS_FIT["fisso"])
    g_f.SetLineWidth(LINE_WIDTH)
    g_f.SetLineStyle(2)
    g_f.Draw("L")

    # One- and two-sigma thresholds (Delta = 1 and Delta = 4).
    linea_1s = ROOT.TLine(intervallo[0], 1.0, intervallo[1], 1.0)
    linea_1s.SetLineColor(COLORS_FIT["soglie"])
    linea_1s.SetLineStyle(2)
    linea_1s.Draw()
    linea_2s = ROOT.TLine(intervallo[0], 4.0, intervallo[1], 4.0)
    linea_2s.SetLineColor(COLORS_FIT["soglie"])
    linea_2s.SetLineStyle(3)
    linea_2s.Draw()

    leg = make_legend(0.20, 0.81, 0.95, 0.91)
    leg.SetNColumns(2)
    leg.AddEntry(g_p, "Others profiled (3D fit)", "l")
    leg.AddEntry(g_f, "Others fixed to 1", "l")
    leg.AddEntry(linea_1s, "1#sigma (#Delta = 1)", "l")
    leg.AddEntry(linea_2s, "2#sigma (#Delta = 4)", "l")
    leg.Draw()

    draw_fcc_label()
    c.Draw()
    tieni_vivi.extend([c, g_p, g_f, linea_1s, linea_2s, leg])


# ============================================================
# Two-dimensional confidence-contour scan for a pair of parameters.
# ============================================================
def scan_2d(var_name, px, py):
    terzo = [p for p in PARAMS if p not in (px, py)][0]
    fit_fisso = fit_results[(var_name, (px, py))]     # terzo parametro fisso a 1
    fit_prof = fit_results[(var_name, PARAMS)]        # terzo parametro profilato
    nome_terzo = LATEX_PARAMETRI[terzo]

    # (fit, confidence level, colour, line style, fill, label).
    # Draw the largest contour first so the axes accommodate all the smaller contours.
    contorni = [
        (fit_prof, 0.95, COLORS_FIT["profilato_chiaro"], 1, True, f"95% CL, {nome_terzo} profiled"),
        (fit_prof, 0.68, COLORS_FIT["profilato"], 1, True, f"68% CL, {nome_terzo} profiled"),
        (fit_fisso, 0.95, COLORS_FIT["fisso_chiaro"], 2, False, f"95% CL, {nome_terzo} = 1"),
        (fit_fisso, 0.68, COLORS_FIT["fisso"], 2, False, f"68% CL, {nome_terzo} = 1"),
    ]

    c = make_canvas(f"c_scan2d_{var_name}_{px}_{py}", "Likelihood Scan 2D")
    c.SetTopMargin(0.20)            # spazio sopra il grafico per la legenda
    c.cd()

    leg = make_legend(0.20, 0.81, 0.95, 0.91)
    leg.SetNColumns(2)
    grafici = []

    for i, (fit, cl, colore, stile, riempi, etichetta) in enumerate(contorni):
        # mncontour returns the (x, y) points of the contour.
        punti = np.array(fit.mncontour(px, py, cl=cl, size=60), dtype=float)
        # mncontour does not close the curve; append the first point to close it.
        x_c = np.append(punti[:, 0], punti[0, 0])
        y_c = np.append(punti[:, 1], punti[0, 1])

        g = ROOT.TGraph(len(x_c), x_c, y_c)
        g.SetLineColor(colore)
        g.SetLineWidth(LINE_WIDTH)
        g.SetLineStyle(stile)
        if riempi:
            g.SetFillColorAlpha(colore, 0.35)
        grafici.append(g)

        opzione = "LF" if riempi else "L"
        if i == 0:
            g.SetTitle(f";{LATEX_PARAMETRI[px]};{LATEX_PARAMETRI[py]}")
            g.Draw("A" + opzione)
            g.GetXaxis().SetTitleOffset(1.4)
            g.GetYaxis().SetTitleOffset(1.4)
        else:
            g.Draw(opzione)
        leg.AddEntry(g, etichetta, "lf" if riempi else "l")

    # Best-fit point.
    g_best = ROOT.TGraph(1, np.array([fit_prof.values[px]], dtype=float),
                            np.array([fit_prof.values[py]], dtype=float))
    g_best.SetMarkerStyle(20)
    g_best.SetMarkerSize(1.2)
    g_best.Draw("P")
    leg.AddEntry(g_best, "Best fit", "p")

    leg.Draw()
    draw_fcc_label()
    c.Draw()
    tieni_vivi.extend([c, leg, g_best] + grafici)


# ============================================================
# Loop over observables, parameters, and parameter pairs.
# ============================================================
# If a scan fails (for example, because the 3D fit is invalid), report it and continue.
for var_name in VARIABILI_SCAN:
    print(f"\n===== {NOMI_VARIABILI[var_name]} =====")

    for param in PARAMS:                            # 3 scan 1D
        try:
            scan_1d(var_name, param)
        except Exception as errore:
            print(f"[!] scan 1D saltato ({var_name}, {param}): {errore}")

    for px, py in combinations(PARAMS, 2):          # 3 scan 2D
        try:
            scan_2d(var_name, px, py)
        except Exception as errore:
            print(f"[!] scan 2D saltato ({var_name}, {px}, {py}): {errore}")

print("\n[OK] Likelihood Scan 1D e 2D generati con successo!")

## 3. Hadronic signal/background BDT

The BDT is trained on the 70% training subsample. The independent 30% holdout subsample is reserved for BDT-score templates and the subsequent likelihood fit.

In [ ]:
# 6. Prepare ROOT trees for TMVA

def crea_tree_per_tmva(nomi_campioni, output_filename):
    """Create a ROOT tree named T with the two observables for the selected training events."""
    f_out = ROOT.TFile.Open(output_path(output_filename), "RECREATE")
    if not f_out or f_out.IsZombie():
        raise OSError(f"Impossibile creare il file ROOT: {output_filename}")

    tree_out = ROOT.TTree("T", "Tree per TMVA")
    cos_star_f = array.array('f', [0.0])
    jet_pt_f = array.array('f', [0.0])
    tree_out.Branch("cos_theta_star", cos_star_f, "cos_theta_star/F")
    tree_out.Branch("jet_pt", jet_pt_f, "jet_pt/F")

    for nome in nomi_campioni:
        for evento in campioni[nome]["train"]:
            cos_star_f[0], jet_pt_f[0] = evento
            tree_out.Fill()

    f_out.cd()
    tree_out.Write()
    f_out.Close()
    print(f"Creato {output_filename}")


# Samples for signal/background BDT training.
crea_tree_per_tmva(["00", "TT", "0T", "T0"], "tmva_adr_signal.root")
crea_tree_per_tmva(["bkg"], "tmva_adr_background.root")

print("\nSample preparation completed.")

In [ ]:
# 7. Train the signal/background BDT
f_signal = ROOT.TFile.Open(output_path("tmva_adr_signal.root"), "READ")
f_bkg = ROOT.TFile.Open(output_path("tmva_adr_background.root"), "READ")
signal_tree = f_signal.Get("T")
bkg_tree = f_bkg.Get("T")

output_file = ROOT.TFile.Open(output_path("TMVA_Adronico_Output.root"), "RECREATE")
factory = ROOT.TMVA.Factory(
    "AnalisiAdr",
    output_file,
    "!V:!Silent:Color:DrawProgressBar:AnalysisType=Classification"
)
dataloader = ROOT.TMVA.DataLoader("dataset_adr")

dataloader.AddVariable("cos_theta_star", "F")
dataloader.AddVariable("jet_pt", "F")
dataloader.AddSignalTree(signal_tree, 1.0)
dataloader.AddBackgroundTree(bkg_tree, 1.0)

# The trees already contain only training events; no additional selection is required.
dataloader.PrepareTrainingAndTestTree(
    ROOT.TCut(""),
    "nTrain_Signal=1000:nTrain_Background=1000:SplitMode=Random:!V"
)

factory.BookMethod(
    dataloader,
    ROOT.TMVA.Types.kBDT,
    "BDT",
    "!H:!V:NTrees=200:MinNodeSize=5%:MaxDepth=3:BoostType=AdaBoost"
)

factory.TrainAllMethods()
factory.TestAllMethods()
factory.EvaluateAllMethods()
output_file.Close()
f_signal.Close()
f_bkg.Close()

print("\nTraining BDT binario completato.")
print("Pesi TMVA: dataset_adr/weights/AnalisiAdr_BDT.weights.xml")

In [ ]:
# 7b. Hadronic BDT output — test sample only

def score_tmva_test(f_tmva, class_id, nome):
    """Build a BDT-score histogram using the TMVA TestTree only."""

    tree = f_tmva.Get("dataset_adr/TestTree")

    if not tree:
        raise KeyError("dataset_adr/TestTree non trovato")

    h = ROOT.TH1F(
        nome,
        ";BDT score;Normalized entries / bin",
        BINS_BDT, -1.0, 1.0
    )
    h.Sumw2()

    n_eventi = 0

    for evento in tree:
        if evento.classID == class_id:
            h.Fill(evento.BDT, evento.weight)
            n_eventi += 1

    print(f"{nome}: {n_eventi} events in the TEST sample")

    h.Rebin(REBIN_BDT)

    if h.Integral() <= 0:
        raise RuntimeError(f"Istogramma vuoto: {nome}")

    h.Scale(1.0 / h.Integral())

    return h


# Open the TMVA output file.
f_tmva_out = ROOT.TFile.Open(
    output_path("TMVA_Adronico_Output.root"), "READ"
)

if not f_tmva_out or f_tmva_out.IsZombie():
    raise OSError("Impossibile aprire TMVA_Adronico_Output.root")


# Test tree only.
h_sig_test = score_tmva_test(
    f_tmva_out, 0, "h_sig_test_only"
)

h_bkg_test = score_tmva_test(
    f_tmva_out, 1, "h_bkg_test_only"
)

f_tmva_out.Close()


# Styling.
style_hist(h_sig_test, COLORS["sig"])
style_hist(h_bkg_test, COLORS["bkg"])


# Canvas..
c_bdt_test_only = make_canvas(
    "c_bdt_test_only",
    "BDT Score - Test Sample"
)

c_bdt_test_only.cd()

max_val = max(
    h_sig_test.GetMaximum(),
    h_bkg_test.GetMaximum()
)

h_sig_test.GetYaxis().SetRangeUser(0.0, max_val * 1.5)

# Plot the test-sample distributions only.
h_sig_test.Draw("HIST E")
h_bkg_test.Draw("HIST E SAME")


# Legend.
leg_bdt_test = make_legend(0.40, 0.74, 0.95, 0.86)

leg_bdt_test.AddEntry(
    h_sig_test, "Signal (test sample)", "lf"
)

leg_bdt_test.AddEntry(
    h_bkg_test, "Background (test sample)", "lf"
)

leg_bdt_test.Draw()

draw_fcc_label()

c_bdt_test_only.RedrawAxis()
c_bdt_test_only.Modified()
c_bdt_test_only.Update()
c_bdt_test_only.Draw()


ROOT_KEEPALIVE.extend([
    c_bdt_test_only,
    leg_bdt_test,
    h_sig_test,
    h_bkg_test
])

print("\n[OK] BDT score: SOLO TEST SAMPLE.")

In [ ]:
# 8. Build BDT templates and perform the likelihood fit

reader = ROOT.TMVA.Reader("!Color:!Silent")
v_cos_star = array.array('f', [0.0])
v_jet_pt = array.array('f', [0.0])
reader.AddVariable("cos_theta_star", v_cos_star)      # stesso nome e stesso ordine del training
reader.AddVariable("jet_pt", v_jet_pt)
reader.BookMVA("BDT", "dataset_adr/weights/AnalisiAdr_BDT.weights.xml")


def punteggio_bdt(evento):
    v_cos_star[0], v_jet_pt[0] = evento
    return reader.EvaluateMVA("BDT")


def histo_bdt(nomi_campioni, nome_hist):
    """Build a BDT-score histogram for the selected samples using holdout events only."""
    h = ROOT.TH1F(nome_hist, f";BDT score;{Y_TITLE}", BINS_BDT, -1.0, 1.0)
    h.Sumw2()
    for nome in nomi_campioni:
        c = campioni[nome]
        h_parte = ROOT.TH1F(f"{nome_hist}_{nome}", "", BINS_BDT, -1.0, 1.0)
        h_parte.Sumw2()
        for evento in c["holdout"]:
            h_parte.Fill(punteggio_bdt(evento))
        h_parte.Scale(c["sigma"] * LUMI_PB / c["n_holdout"])
        h.Add(h_parte)
    return h


h_bdt = {
    "00": histo_bdt(["00"], "h_bdt_00"),
    "TT": histo_bdt(["TT"], "h_bdt_tt"),
    "mixed": histo_bdt(["0T", "T0"], "h_bdt_mixed"),
    "bkg": histo_bdt(["bkg"], "h_bdt_bkg"),
}

# Rebin by combining REBIN_BDT adjacent bins. This increases the number of events per template bin
# and reduces the relative statistical uncertainty; Sumw2 propagates errors in quadrature.
for h_template in h_bdt.values():
    h_template.Rebin(REBIN_BDT)
N_BINS_FIT = h_bdt["bkg"].GetNbinsX()

# --- Plot the templates normalized to the integrated luminosity ---
c_tb = make_canvas("c_bdt_template", "Template del BDT")
c_tb.cd()

for chiave in ["00", "TT", "mixed", "bkg"]:
    style_hist(h_bdt[chiave], COLORS[chiave], fill=(chiave != "bkg"))

massimo = max(h.GetMaximum() for h in h_bdt.values())
h_bdt["00"].GetYaxis().SetRangeUser(0.0, massimo * 1.5)
h_bdt["00"].GetYaxis().SetTitleOffset(1.4)
h_bdt["00"].Draw("HIST E")
for chiave in ["TT", "mixed", "bkg"]:
    h_bdt[chiave].Draw("HIST E SAME")

leg_tb = make_legend(0.20, 0.66, 0.95, 0.85)
for chiave in ["00", "TT", "mixed", "bkg"]:
    leg_tb.AddEntry(h_bdt[chiave], f"{LABELS[chiave]} [Int: {h_bdt[chiave].Integral():.3e}]", "l")
leg_tb.Draw()
draw_fcc_label()
c_tb.Draw()

# Perform a three-parameter likelihood fit to the BDT score using the functions defined above.
template_bdt = {
    "bkg": np.array([h_bdt["bkg"].GetBinContent(i) for i in range(1, N_BINS_FIT + 1)], dtype=float),
    "00": np.array([h_bdt["00"].GetBinContent(i) for i in range(1, N_BINS_FIT + 1)], dtype=float),
    "tt": np.array([h_bdt["TT"].GetBinContent(i) for i in range(1, N_BINS_FIT + 1)], dtype=float),
    "mixed": np.array([h_bdt["mixed"].GetBinContent(i) for i in range(1, N_BINS_FIT + 1)], dtype=float),
}
fit_bdt = esegui_fit_likelihood(template_bdt, PARAMS)

print("\n========== BDT LIKELIHOOD FIT (Asimov data) ==========")
for p in PARAMS:
    print(f"{NOMI_PARAMETRI[p]} = {fit_bdt.values[p]:.5f} ± {fit_bdt.errors[p]:.5f}")
print(f"fit valid: {fit_bdt.valid}")
for p1, p2 in combinations(PARAMS, 2):
    print(f"ρ({NOMI_PARAMETRI[p1]}, {NOMI_PARAMETRI[p2]}) = {leggi_correlazione(fit_bdt, p1, p2, PARAMS):+.4f}")

In [ ]:
# Save all canvases as vector PDF and PNG files
# PDF preserves vector quality when figures are resized.
# Run this cell after creating the canvases to export all figures.
import os
for canvas in ROOT.gROOT.GetListOfCanvases():
    canvas.SaveAs(output_path(f"plots/{canvas.GetName()}.pdf"))
    canvas.SaveAs(output_path(f"plots/{canvas.GetName()}.png"))
